In [7]:
from qiskit import QuantumCircuit
from qforte import SQOpPool, Circuit
from qforte.system import system_factory
from qforte.evolution.trotter import trotter_evolve, hartree_fock
from qforte.qiskit_api.translators import qforte_to_qiskit
from qforte.qiskit_api.dispatchers import AerDispatcher
import numpy as np
import cmath

geom = [
    ('H', (0., 0., 1.0)), 
    ('H', (0., 0., 2.0))
    ]

mol = system_factory(
    build_type='psi4',
    mol_geometry=geom,
    basis='sto-3g',
    df_icut=1.0e-6)

#number of krylov basis states to generate
krylov_basis_size = 3
#timestep size (seconds)
dt = 0.5


#computational backend to use with Qiskit
aer_dispatcher = AerDispatcher();
#number of samples
shots = 10000

#construct the hartree fock and zero states for H2
hf_state = hartree_fock(mol, QuantumCircuit(4))
zero_state = QuantumCircuit(4)

#create a circuit to prepare the GHZ state on 4 qubits
ghz_circ = QuantumCircuit(4)
ghz_circ.h(0)
ghz_circ.cx(0, 1)

#prepare the GHZ state by applying the GHZ circuit to the zero state
ghz_state = zero_state.compose(ghz_circ)

#construct the 1st order trotterized evolution circuit for H2
qf_trotter_circ, qf_trotter_computer = trotter_evolve(mol, dt)
qiskit_trotter_circ = qforte_to_qiskit(qf_trotter_circ, qf_trotter_computer.get_nqubit())

#get the first krylov basis state by applying the trotterized evolution circuit to both the hartree fock state and the GHZ state
hf_krylov_states = [hf_state.compose(qiskit_trotter_circ)]
ghz_krylov_states = [ghz_state.compose(qiskit_trotter_circ)]

#get the rest of the krylov basis states by repeatedly applying the trotterized evolution circuit to both the hartree fock state and the GHZ state
for i in range(krylov_basis_size - 1):
    hf_krylov_states.append(hf_krylov_states[-1].compose(qiskit_trotter_circ))
    ghz_krylov_states.append(ghz_krylov_states[-1].compose(qiskit_trotter_circ))

#apply the inverse of the GHZ circuit to all the GHZ-based krylov states to map the GHZ state back to the zero state
ghz_krylov_states = [state.compose(ghz_circ.inverse()) for state in ghz_krylov_states]

#append measurements to all the circuits
for state in hf_krylov_states:
    state.measure_all()
for state in ghz_krylov_states:
    state.measure_all()
    
#dispatch the circuits to aer for sampling
aer_dispatcher = AerDispatcher(); shots = 10000
hf_krylov_results = aer_dispatcher.dispatch_sampler(hf_krylov_states, shots)
ghz_krylov_results = aer_dispatcher.dispatch_sampler(ghz_krylov_states, shots)

#extract the sample counts from Qiskit's data structures
hf_krylov_samples = [res.data.meas.get_counts() for res in hf_krylov_results]
ghz_krylov_samples = [res.data.meas.get_counts() for res in ghz_krylov_results]

#calculate F1 for each krylov state by taking the count of the '0011' bitstring and dividing by the total number of shots
F1 = [sample["0011"]/shots for sample in hf_krylov_samples]
print(f"\nF1 values for {krylov_basis_size} krylov states with dt = {dt} seconds: {F1}")

#calculate F2 for each krylov state by taking the count of the '0000' bitstring (mapped to the GHZ state) and dividing by the total number of shots
F2 = [sample["0000"]/shots for sample in ghz_krylov_samples]
print(f"\nF2 values for {krylov_basis_size} krylov states with dt = {dt} seconds: {F2}")

#calculate r values by taking the square root of F1 for each krylov state
r = [f1_val**0.5 for f1_val in F1]
print(f"\nr values for {krylov_basis_size} krylov states with dt = {dt} seconds: {r}")

nuclear_repulsion_energy = mol.nuclear_repulsion_energy
theta_r = [(-dt * nuclear_repulsion_energy * (i+1)) for i in range(krylov_basis_size)]
print(f"\ntheta_r values for {krylov_basis_size} krylov states with dt = {dt} seconds: {theta_r}")

theta = [(np.acos((4*f2 - f1 - 1) / (2*np.sqrt(f1)))+ th_r) for f1, f2, th_r in zip(F1, F2, theta_r)]
print(f"\ntheta values for {krylov_basis_size} krylov states with dt = {dt} seconds: {theta}")

states = [cmath.rect(r_val, theta_val) for r_val, theta_val in zip(r, theta)]
print(f"\nStates for {krylov_basis_size} krylov states with dt = {dt} seconds: {states}")

 ==> Psi4 geometry <==
-------------------------
0  1
H  0.0  0.0  1.0
H  0.0  0.0  2.0
symmetry c1
units angstrom

F1 values for 3 krylov states with dt = 0.5 seconds: [0.9915, 0.9602, 0.9268]

F2 values for 3 krylov states with dt = 0.5 seconds: [0.8502, 0.4802, 0.1243]

r values for 3 krylov states with dt = 0.5 seconds: [0.9957409301620578, 0.9798979538707079, 0.9627045237246992]

theta_r values for 3 krylov states with dt = 0.5 seconds: [-0.264588605335, -0.52917721067, -0.7937658160050001]

theta values for 3 krylov states with dt = 0.5 seconds: [np.float64(0.5200212344882391), np.float64(1.0617246048738336), np.float64(1.6138128157385414)]

States for 3 krylov states with dt = 0.5 seconds: [(0.864112570931863+0.49478223973534674j), (0.47756981542726684+0.8556442434755033j), (-0.04139939804317407+0.9618139580197735j)]


In [ ]:
# find the expectation value of the hamiltonian with respect to the vacuum from the sq hamiltonian
sq_hamiltonian = mol.sq_hamiltonian
naive_qb_hamiltonian = mol.sq_hamiltonian.jw_transform()
hamiltonian_circ = Circuit()

def jw_transform_hermitian(hamiltonian):
    sum = []
    
    hermitian_pairs = SQOpPool()
    hermitian_pairs.add_hermitian_pairs(1.0, hamiltonian)
    for term in hermitian_pairs:
        coeff = term[0]
        op = term[1]
        op = op.jw_transform()
        op.mult_coeffs(coeff)
        for term in op.terms():
            sum.append(term)

    return sum

#get the qubit hamiltonian
hermitian_qb_hamiltonian = jw_transform_hermitian(sq_hamiltonian)
vacuum_expectation_value = 0.0
for term in hermitian_qb_hamiltonian:
    print(term)
    if "X" in term[1].str() or "Y" in term[1].str():
        print("non-diagonal term found")
    else:
        vacuum_expectation_value += term[0]
    

#compare the results with the known nuclear repulsion energy
print(f"\nVacuum expectation value: {vacuum_expectation_value}")
nuclear_repulsion_energy = mol.nuclear_repulsion_energy
print(f"\nNuclear repulsion energy: {nuclear_repulsion_energy}")


((0.52917721067+0j), [])
((-0.555422089825263+0j), [])
((0.555422089825263+0j), [Z0])
((-0.555422089825263+0j), [])
((0.555422089825263+0j), [Z1])
((0.15660062486143403+0j), [])
((-0.15660062486143403-0j), [Z0])
((-0.15660062486143403-0j), [Z1])
((0.15660062486143403+0j), [Z1 Z0])
((0.024598822939426418+0j), [X3 X2 X1 X0])
non-diagonal term found
((-0.024598822939426418+0j), [Y3 Y2 X1 X0])
non-diagonal term found
((0.024598822939426418+0j), [Y3 X2 Y1 X0])
non-diagonal term found
((0.024598822939426418+0j), [X3 Y2 Y1 X0])
non-diagonal term found
((0.024598822939426418+0j), [Y3 X2 X1 Y0])
non-diagonal term found
((0.024598822939426418+0j), [X3 Y2 X1 Y0])
non-diagonal term found
((-0.024598822939426418+0j), [X3 X2 Y1 Y0])
non-diagonal term found
((0.024598822939426418+0j), [Y3 Y2 Y1 Y0])
non-diagonal term found
((-0.2945605019225245+0j), [])
((0.2945605019225245+0j), [Z2])
((0.10622904488350782+0j), [])
((-0.10622904488350782+0j), [Z0])
((-0.10622904488350782+0j), [Z2])
((0.10622904488350

In [22]:
-dt*2*nuclear_repulsion_energy

-0.52917721067